# Setup


In [ ]:
import pandas as pd
from pathlib import Path
import requests
import os
from tqdm import tqdm
import yaml
from requests.adapters import HTTPAdapter, Retry
from concurrent.futures import ThreadPoolExecutor
import threading

def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")

def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None





PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_PATH_META = PROCESSED_DIR / "metadata.parquet"
IMG_PATH = PROJECT_ROOT / "data" / "raw" / "images"

IMG_PATH.mkdir(
    parents = True,
    exist_ok = True
)

# Parameter: versioniert, damit alle im Team denselben Datensatz erzeugen.
cfg_file = find_upwards("config.yaml")
assert cfg_file, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(cfg_file.read_text())

MAX_WORKERS = CFG["max_workers"]
if MAX_WORKERS == "auto":
    MAX_WORKERS = min(16, (os.cpu_count() or 4) * 2)
else:
    MAX_WORKERS = int(MAX_WORKERS)
    
IMG_SIZE = CFG["image_size"]

metadata = pd.read_parquet(DATA_PATH_META)



# Token: geheim, deshalb bewusst nicht in config.yaml.
if env_file := find_upwards(".env"):
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))


TOKEN = os.environ.get("MAPILLARY_TOKEN", "")

assert TOKEN.startswith("MLY|"), (
    "Kein Mapillary-Token. Datei .env anlegen:\n    MAPILLARY_TOKEN=MLY|dein|token"
)




# API Anfrage


In [ ]:
def make_session():
    session = requests.Session()
    
    retry = Retry(
        total = 5, backoff_factor = 0.6, status_forcelist =[429, 500, 502, 503, 504], allowed_methods = ["GET"], respect_retry_after_header = True
    )
    
    adapter = HTTPAdapter(max_retries = retry, pool_connections = MAX_WORKERS, pool_maxsize = MAX_WORKERS)
    
    session.mount("https://", adapter)
    
    return session

thread_local = threading.local()

def get_session():
    
    if not hasattr(thread_local, "session"):
        thread_local.session = make_session()
        
    return thread_local.session


def download_image(image_id):

    image_id = str(image_id)
    image_file = IMG_PATH / f"{image_id}.jpg"
    
    if image_file.exists():
        return "exists"
    

    try:
        URL = f"https://graph.mapillary.com/{image_id}"

        params = {
            "fields": f"id,thumb_{IMG_SIZE}_url",
            "access_token": TOKEN
        }

        response = requests.get(URL, params = params, timeout = 30)
        response.raise_for_status()
        
        data = response.json()

        image_url = data.get(f"thumb_{IMG_SIZE}_url")
        
        if not image_url:
            print(f"Keine Bild URL {image_id}")
            return "failed"
        
        
        image_response = requests.get(
            image_url,
            timeout = 60
        )
        
        image_response.raise_for_status()
        
        image_file.write_bytes(
            image_response.content
        )
        
        return "downloaded"
    
    except requests.RequestException as e:
        print(f"Fehler bei {image_id}: {e}")
        return "failed"
    
    except Exception as e:
        print(f"Anderer Fehler bei {image_id}: {e}")
        return "failed"
    
    

In [ ]:

'''
#Test only 1 Picture:
test_image_id = str(metadata.iloc[1]["image_id"])
result = download_image(test_image_id)
print("Ergebnis 1 Bild: ", result)
'''

# Wie viele Bilder müssen geladen werden
download_metadata = metadata[
    metadata["split"].isin(["database", "query"])
].copy()

print(
    f"Zu ladende Bilder: "
    f"{len(download_metadata):,}"
)

# Download

results = []

with ThreadPoolExecutor(max_workers = MAX_WORKERS) as executor:
    
    for result in tqdm (
        executor.map(download_image, download_metadata["image_id"]),
        total = len(download_metadata),
        desc = "Bilder herunterladen"
    ):
        results.append(result)
        


# Auswertung erster Download
print("=" * 50)
print("DOWNLOAD ERGEBNISSE")
print("=" * 50)

print(f"Erfolgreiche:       {results.count('downloaded'):,}")
print(f"Bereits vorhanden:  {results.count('exists'):,}")
print(f"Fehlgeschlagene:    {results.count('failed'):,}")
print("=" * 50)


failed_ids = [image_id for image_id, result in zip(download_metadata["image_id"], results) if result == "failed"]

print(f"Fehlgeschalgende Bilder:    {len(failed_ids):,}")

FAILED_IMG_PATH = ( PROCESSED_DIR / "failed_image_download.txt")
FAILED_IMG_PATH.write_text("\n".join(map(str, failed_ids)))

print(f"Gepeicherte failed unter {FAILED_IMG_PATH}")

# Count *jpg 
image_files = list(IMG_PATH.glob("*.jpg"))

print(f"Lokale JPGs: {len(image_files):,}")